# Attention Residuals

This is a small-scale replication of [Attention Residual](https://arxiv.org/abs/2603.15031) on a single v6 TPU. This will create, run and benchmark standard Kimi Linear against Kimi Linear with Attention Residuals(Full/Block) with identical configurations.

Specifications:
- 300M Param Model
- 256-512 Batch Size
- 1024-2048 Context Length


There will be a Kimi Linear baseline and a (+ AttnRes) variant. They will be trained individually across a single v6 TPU using jax, looking to verify and see the supposed benefit of AttnResiduals from the paper.


Pre-training occurs using 4090-context window, Muon[1] and a WSG(Warmup-Stable-Decay) learning rate schedule with a gloabl batch size of 8M tokens.

> *"Our architecture is identical to Kimi Linear ...., which interleaves Kimi Delta Attention (KDA) and Multi-Head Latent Attention (MLA) layers in a 3:1 ratio, each followed by an MoE feed-forward layer. The only modification is the addition of AttnRes to the residual connections; all other components ... remain unchanged."* - *__Authors__*

[1]: https://kellerjordan.github.io/posts/muon/



*Question*: **Are Attention Residuals findings consistent across single tru training? Furthermore, can a 300M parameter MoE Tranfsormer with Attention Residuals bring lower validation loss compared to PreNorm? And as an extension, how can it best be optimized for single device training?**

Some questions I also explore in this small-scale experimentation sprint:
1. Do deeper layers attend to earlier layers?
2. Does the distribution flatten or sharpen during training?
3. Is AttenRes using many layers or colapsing into one instead(high vs low entropy)?

Further things to possibly explore: Muon vs AdamW vs Prodigy | WSD vs Cosine | What if Attention Residual per Top and Bottom Block instead of between each attn and moe layer?

In [4]:
!uv pip install jax flax huggingface_hub huggingface

Resolved 51 packages in 386ms                                        
⠙ Preparing packages... (0/3)                                                   
⠙ Preparing packages... (0/3)--------------     0 B/26.13 KiB           
⠙ Preparing packages... (0/3)2m------------ 14.88 KiB/26.13 KiB         
⠙ Preparing packages... (0/3)2m------------ 14.88 KiB/26.13 KiB         
filelock             ------------------------------ 14.88 KiB/26.13 KiB
⠙ Preparing packages... (0/3)--------------     0 B/603.55 KiB          
filelock             ------------------------------ 26.13 KiB/26.13 KiB
⠙ Preparing packages... (0/3)--------------     0 B/603.55 KiB          
filelock             ------------------------------ 26.13 KiB/26.13 KiB
⠙ Preparing packages... (0/3)--------------     0 B/603.55 KiB          
filelock             ------------------------------ 26.13 KiB/26.13 KiB
⠙ Preparing packages... (0/3)--------------     0 B/603.55 KiB          
filelock             -----------------------------

In [7]:
import jax
import optax
import flax
import jax.lax as lax
import jax.numpy as jnp
import huggingface_hub
import functools
from typing import Optional, Tuple, Literal, NamedTuple, Any
import flax.linen as nn
from einops import rearrange



# Kimi Linear


Kimi Linear components in the cell below:

1. Kimi Delta Attention (chunked KDA)
2. Diagonal Plus Low Rank (DPLR)
3. Multi-head Latent Attention (MLA)

Some Others:
1. Utils
2. Linear
3. RoPE
4. Muon Optimizer

As well as the main AttnRes which is implemented in the block below.



# Experiment

## MoE

The transformer is not carbon-copy of DeepSeek v3 nor fully represent the experiment configurations of Residual Attention. Rather it is representative of compute capabilities, as reflected by differences in context window, token size, and architectural decisions further explained below. It is, to it's best however, aligned with the paper.


## Differences

1. Column & Row Parallel Linear don't exist
2. Parallel Embeddings don't exist
3. Only contains Naive KDA




In [ ]:
from jax._src.lax.parallel import psend

block_size: int = 128

class ModelArg:
  """

  Model Arguments for Kimi Linear

  Configurations for training a MoE Transformer, aligned closely
  with [Deepseek et al](https://arxiv.org/abs/2502.16982)_.. It further
  is configured to run as efficent as possible on a single v6 TPU.

  This is configured to have a baseline kimi linear MoE with a swapable
  PreNorm, to train and benchmark full and block attention residuals.




  """
  score_func: Literal['softmax', 'sigmoid'] = 'softmax'
  inter_dim: int = 2048
  dim: int = 1024
  in_feat: int
  out_feat: int
  grouped_experts: int
  active_experts: int = 64
  experts: int = 256
  shared_experts: int = 1
  block_size: int
  scaling_factor: float =  2.446




# Kimi Linear's KDA Forward


# Utils

def exists(x):
  return x is not None

def default(val, d):
  if exists(val):
    return val
  return d() if callable(d) else d



# PreNorm

class PreNorm(nn.Module):
  d_model: int

  @nn.compact
  def __call__(self, x):
    value = nn.RMSNorm()(x)
    out = nn.Dense(self.d_model)(value)
    return x + out


# RMSNorm

class RMSNorm(nn.Module):
  """
  Root Mean Squared Normalization
  """
  dim: int
  eps: float = 1e-6

  def setup(self):
      self.weight = self.param('weight', nn.initializers.ones, (self.dim,))

  def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
      rms = jnp.sqrt(jnp.mean(x ** 2, axis=-1, keepdims=True) + self.eps)
      return (x / rms) * self.weight

# Linear(s)

def linear(x: jnp.ndarray, weights: jnp.ndarray, bias: Optional[jnp.ndarray] = None) -> jnp.ndarray:
  if weights.itemsize > 1:
    return jnp.dot(x, weights)
  if exists(bias):
    return jnp.dot(x, weights) + bias
  return jnp.dot(x, weights)


class Linear(nn.Module):
  in_features: int
  out_features: int
  use_bias: bool = False

  @nn.compact
  def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
    """
    Forward call for a linear layer
    """
    weights = self.param(
        'weights',
        nn.initializers.zeros,
        (self.in_features, self.out_features)
    )
    bias = None
    if self.use_bias:
      bias = self.param(
          'bias',
          nn.initializers.zeros,
          (self.out_features,)
      )
    return linear(x, weights, bias)



# Short Convolution

class ShortConv(nn.Module):
  """
  Short Convolution function
  """

  @nn.compact
  def __call__(self, x: jnp.ndarray, weights: jnp.ndarray, bias: jnp.ndarray, k: int = 4) -> jnp.ndarray:
    """
    Forward call for a short convolution
    """

    batch, seq, dim = x.shape

    x_padded = jnp.pad(x, ((0,0), (k-1, 0), (0,0)))

    out = jnp.zeros((batch, seq, dim))
    for i in range(k):
        out = out + x_padded[:, i:i+seq, :] * weights[None, None, :, i].squeeze(-1)

    return out + bias


# Row Parallel Linear

# Chunked Kimi Delta Attention

class KDA(nn.Module):
  """
  Chunked Kimi Delta Attention
  """
  dim: int
  heads: int
  head_dim: int = 128
  chunk_size: int = 64

  @nn.compact
  def __call__(self,
               q: jnp.ndarray,
               k: jnp.ndarray,
               v: jnp.ndarray,
               g: jnp.ndarray,
               beta: jnp.ndarray,
               initial_state: Optional[jnp.ndarray] = None,
               ) -> tuple[jnp.ndarray, jnp.ndarray]:

      B, T, H, K = q.shape
      V = v.shape[-1]
      C = self.chunk_size
      N = T // C
      assert T % C == 0


      q, k, v, g, beta = [x.astype(jnp.bfloat16) for x in (q, k, v, g, beta)]


      q, k, v, g = [rearrange(x, 'b (n c) h d -> b h n c d', c=C)
                    for x in (q, k, v, g)]
      beta = rearrange(beta, 'b (n c) h -> b h n c', c=C)

      q = q * (K ** -0.5)
      g = jnp.cumsum(g, axis=-2)


      A = jnp.einsum('...jd,...id->...ji',
                      k * jnp.exp(g),
                      k * jnp.exp(-g),
                      precision=jax.lax.Precision.HIGHEST)

      A = A * beta[..., None]


      mask_diag = jnp.tril(jnp.ones((C, C), dtype=jnp.bool_))
      A = -jnp.where(mask_diag, A, 0.0)


      def fwd_sub_step(i, A):
          col    = A[..., i:i+1, :]
          update = (col * A).sum(-2)
          return A.at[..., i, :].add(update)

      A = jax.lax.fori_loop(1, C, fwd_sub_step, A)


      A = (A + jnp.eye(C, dtype=jnp.bfloat16)) * beta[..., None, :]


      w = jnp.einsum('...ij,...jd->...id', A, jnp.exp(g) * k,
                      precision=jax.lax.Precision.HIGHEST)
      u = jnp.einsum('...ij,...jd->...id', A, v,
                      precision=jax.lax.Precision.HIGHEST)

      S_init = jnp.zeros((B, H, K, V), dtype=jnp.bfloat16)
      if initial_state is not None:
          S_init = S_init + initial_state.astype(jnp.bfloat16)

      mask_strict = jnp.triu(jnp.ones((C, C), dtype=jnp.bool_), k=1)

      def chunk_step(S, inputs):
          q_i, k_i, u_i, g_i, w_i = inputs


          A_qk = jnp.einsum('...id,...jd->...ij',
                             q_i * jnp.exp(g_i),
                             k_i * jnp.exp(-g_i),
                             precision=jax.lax.Precision.HIGHEST)
          A_qk = jnp.where(mask_strict, 0.0, A_qk)


          v_i = u_i - jnp.einsum('...id,...dv->...iv', w_i, S,
                                  precision=jax.lax.Precision.HIGHEST)


          o_i = (jnp.einsum('...id,...dv->...iv',
                             q_i * jnp.exp(g_i), S,
                             precision=jax.lax.Precision.HIGHEST)
               + jnp.einsum('...ij,...jv->...iv',
                             A_qk, v_i,
                             precision=jax.lax.Precision.HIGHEST))


          decay = g_i[..., -1:, :]
          S_new = (S * jnp.exp(rearrange(decay, 'b h 1 k -> b h k 1'))
                 + jnp.einsum('...id,...iv->...dv',
                              k_i * jnp.exp(-g_i), v_i,
                              precision=jax.lax.Precision.HIGHEST))

          return S_new, o_i


      S_final, o = jax.lax.scan(
          chunk_step,
          S_init,
          (q, k, u, g, w)
      )

      return (
          rearrange(o, 'b h n c v -> b (n c) h v').astype(jnp.bfloat16),
          S_final
      )






# Muon - Optimizer

def newton_schulz(g: jnp.ndarray, steps: int = 5, eps: float = 1e-7):
  """
  Newton-Schultz iteration on 2D matrix
  """
  assert g.ndim >= 2

  a, b, c = 3.4445, -4.7750, 2.0315

  x = g.astype(jnp.float32)

  x = x / (jnp.linalg.norm(x, axis=(-2, -1), keepdims=True) + eps)

  def _ns_step(_, x):
      A = x @ x.T
      B = b * A + c * A @ A
      return a * x + B @ x

  x = jax.lax.fori_loop(0, steps, _ns_step, x)

  if g.shape[-2] > g.shape[-1]:
      x = x.T

  return x

class MuonState(NamedTuple):
    momentum_buffer: Any
    count: jnp.ndarray


def init_muon(params: Any) -> MuonState:
    return MuonState(
        momentum_buffer=jax.tree_util.tree_map(
            lambda p: jnp.zeros_like(p, dtype=jnp.bfloat16), params
        ),
        count=jnp.zeros([], jnp.int32),
    )


def muon_step(
    grads: Any,
    state: MuonState,
    learning_rate: float = 0.02,
    beta: float = 0.95,
    nesterov: bool = True,
    ns_steps: int = 5,
) -> tuple[Any, MuonState]:
    results = jax.tree_util.tree_map(
        lambda g, b: muon_update(g, b, beta=beta, ns_steps=ns_steps, nesterov=nesterov),
        grads, state.momentum_buffer,
    )
    updates = jax.tree_util.tree_map(lambda r: learning_rate * r[0], results)
    new_buf = jax.tree_util.tree_map(lambda r: r[1], results)

    return updates, MuonState(momentum_buffer=new_buf, count=state.count + 1)



def muon_update(
    grad: jnp.ndarray,
    buf: jnp.ndarray,
    beta: float = 0.95,
    ns_steps: int = 5,
    nesterov: bool = True,
) -> tuple[jnp.ndarray, jnp.ndarray]:
    g = grad.astype(jnp.float32)
    b = buf.astype(jnp.float32)

    new_buf = beta * b + (1.0 - beta) * g
    update = beta * new_buf + (1.0 - beta) * g if nesterov else new_buf

    if grad.ndim == 2:
        update = newton_schulz(update, steps=ns_steps)
        m, n = update.shape
        update = update * max(1, m / n) ** 0.5
    elif grad.ndim >= 3:
        update = jax.vmap(functools.partial(newton_schulz, steps=ns_steps))(update)
        m, n = update.shape[-2], update.shape[-1]
        update = update * max(1, m / n) ** 0.5

    return update.astype(grad.dtype), new_buf.astype(buf.dtype)



# Multi Linear Attention

class multihead_attn(nn.Module):
  """
  Multi-head latent Attention
  """
  dim: int
  heads: int
  local_heads: int
  q_lora_rank: int
  kv_lora_rank: int
  qk_head_dim: Optional[int] = None
  v_head_dim: Optional[int] = None


  def setup(self, args: ModelArg):
    self.dim = args.dim
    self.heads: int = args.heads
    self.local_heads: int = args.local_heads
    self.q_lora_rank: int = args.q_lora_rank
    self.kv_lora_rank: int = args.kv_lora_rank
    if self.q_lora_rank == 0:
      self.wq = Linear(self.dim, self.heads * self.qk_head_dim)
    else:
      self.wq_a = Linear(self.dim, self.q_lora_rank)
      self.q_lora = nn.RMSNorm(self.q_lora_rank)
      self.wq_b = Linear(self.q_lora_rank, self.heads * self.qk_head_dim)

    self.wkv_a = Linear(self.dim, self.kv_lora_rank + self.qk_rope_head_dim)
    self.kv_norm = nn.RMSNorm(self.kv_lora_rank)
    self.wkv_b = Linear(self.kv_lora_rank, self.heads * (self.qk_nope_head_dim + self.v_head_dim))
    self.wo = Linear(self.heads * self.v_head_dim, self.dim)



  @nn.compact
  def __call__(self, x: jnp.ndarray, start: int, freqs_cis: jnp.ndarray, mask: Optional[jnp.ndarray] = None):
    """
    Forward call for a Multi-head Attention
    """

    assert x.dtype == jnp.float16, "Input must be float16"

    batch, seqlen, _ = x.shape
    end = seqlen + start

    if self.q_lora_rank is None:
      q = self.wq(x)
    else:
      q = self.wq_b(self.q_norm(self.wq_a(x)))

    q = q.reshape(batch, seqlen, self.local_heads, self.qk_head_dim)

    q_nope, q_pe = jnp.split(q, [self.qk_nope_head_dim], axis=-1)



    kv = self.wkv_a(x)

    kv, k_pe = jnp.split(kv, [self.kv_lora_rank], axis=-1)


    # This only has naive MLA variant

    q = jnp.concatenate([q_nope, q_pe], axis=-1)

    kv = self.wkv_b(self.kv_norm(kv))

    kv = kv.reshape(batch, seqlen, self.local_heads, self.qk_nope_head_dim + self.v_head_dim)

    k_nope, v = jnp.split(kv, [self.qk_nope_head_dim], axis=-1)

    k = jnp.concatenate([k_nope, jnp.broadcast_to(k_pe[:, :, None, :], k_nope.shape)], axis=-1)

    self.k_cache = self.k_cache.at[:batch, start:end].set(k)

    self.v_cache = self.v_cache.at[:batch, start:end].set(v)

    scores = jnp.einsum("bshd,bthd->bsht", q, self.k_cache[:batch, :end]) * self.softmax_scale

    if mask is not None:
      mask = mask[None, :,  :]
      scores += mask
    scores = jax.nn.softmax(scores.astype(jnp.float32), axis=-1).astype(x.dtype)
    x = jnp.einsum("bsht,bthd->bshd", scores, self.v_cache[:batch, :end])

    x = self.wo(x.reshape(batch, seqlen, -1))
    return x


# Gate - Routes to the right MoE

class Gate(nn.Module):
  """
  Gate - Mechanism that routes to the right MoE Expert

  Routes tokens to top-k experts via 2-stage hierarchical selection:
    1. Score all experts via sigmoid/softmax
    2. Select top-k groups by top-2 expert score sum per group
    3. Select top-k experts from surviving groups
  Bias is per-expert for auxiliary-loss-free load balancing (DeepSeek V3).
  """
  dim: int
  n_routed_experts: int
  topk: int
  n_groups: int
  topk_groups: int
  route_scale: float
  score_func: str = 'sigmoid'



  @nn.compact
  def __call__(self, x: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:

    weight = self.param(
        'weight',
        nn.initializers.normal(stddev=0.01),
        (self.n_routed_experts, self.dim)
    )

    e_score_correction_bias = self.param(
        'e_score_correction_bias',
        nn.initializers.zeros,
        (self.n_routed_experts,)
    )

    scores = linear(x, weight.T)

    if self.score_func == "softmax":
      scores = jax.nn.softmax(scores, axis=-1).astype(jnp.float32)
    else:
      scores = jax.nn.sigmoid(scores)

    original_scores = scores

    # Add per-expert bias for routing decisions only
    scores = scores + e_score_correction_bias

    if self.n_groups > 1:
      scores = scores.reshape(x.shape[0], self.n_groups, -1)

      # Group score: sum of top-2 expert scores within each group
      group_score = jnp.sort(scores, axis=-1)[..., -2:].sum(axis=-1)

      group_idx = jnp.argsort(group_score, axis=-1)[..., -self.topk_groups:]

      mask = jnp.ones((x.shape[0], self.n_groups), dtype=bool)
      mask = mask.at[jnp.arange(x.shape[0])[:, None], group_idx].set(False)

      scores = jnp.where(mask[..., None], -jnp.inf, scores).reshape(x.shape[0], -1)

    # Select top-k experts
    indices = jnp.argsort(scores, axis=-1)[..., -self.topk:]

    # Final weights from original (un-biased) scores
    weights = original_scores[jnp.arange(x.shape[0])[:, None], indices]

    if self.score_func == "sigmoid":
      weights = weights / (weights.sum(axis=-1, keepdims=True) + 1e-20)

    weights = weights * self.route_scale

    return weights, indices


# Expert - The Expert in MoE

class Expert(nn.Module):
  dim: int
  inter_dim: int


  @nn.compact
  def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
    """
    Forward pass for the Expert layer
    """
    w1 = Linear(self.dim, self.inter_dim)
    w2 = Linear(self.inter_dim, self.dim)
    w3 = Linear(self.dim, self.inter_dim)


    return w2(nn.silu(w1(x)) * w3(x))

# MLP - Mulit-Layer Perceptron


class MLP(nn.Module):
  dim: int
  inter_dim: int

  @nn.compact
  def __call__(self, x: jnp.ndarray):
    w1 = Linear(self.dim, self.inter_dim)
    w2 = Linear(self.inter_dim, self.dim)
    w3 = Linear(self.dim, self.inter_dim)

    return w2(jax.nn.silu(w1(x)) * w3(x))


# MoE - Mixture of Experts Layer

class MixtureOfExperts(nn.Module):

    dim: int
    n_routed_experts: int
    n_groups: int
    n_activated_experts: int
    n_shared_experts: int
    topk_groups: int
    inter_dim: int
    route_scale: float
    score_func: str


    def setup(self):
        self.gate = Gate(dim=self.dim, n_routed_experts=self.n_routed_experts,
                         topk=self.n_activated_experts,
                         n_groups=self.n_groups, topk_groups=self.topk_groups,
                         route_scale=self.route_scale, score_func=self.score_func
                         )
        # Vmapped expert: params stacked along axis 0, single call for all experts
        self.experts = nn.vmap(
            Expert,
            variable_axes={'params': 0},
            split_rngs={'params': True},
            in_axes=None, out_axes=0,
            axis_size=self.n_routed_experts
        )(self.dim, self.inter_dim)

        self.shared_experts = MLP(self.dim, self.n_shared_experts * self.inter_dim)

    def __call__(self, x: jnp.ndarray):
        shape = x.shape
        x = x.reshape(-1, self.dim)

        weights, indices = self.gate(x)

        # All experts in parallel: (n_experts, n_tokens, dim)
        all_expert_out = self.experts(x)

        # Gather selected expert outputs: (n_tokens, topk, dim)
        selected_out = all_expert_out[indices.T].transpose(1, 0, 2)  # TODO: verify indexing

        # Weighted sum over topk: (n_tokens, dim)
        y = (selected_out * weights[..., None]).sum(axis=1)

        z = self.shared_experts(x)
        return (y + z).reshape(shape)


# Block - Transformer Block combining KSA and FFN

class Block(nn.Module):
  """
  Transformer Block
  """
  args: ModelArg
  layer_id: int


  def setup(self):
    self.attention = multihead_attn(
        self.args.dim,
        self.args.head,
        self.args.local_heads,
        self.args.q_lora_rank,
        self.args.kv_lora_rank,
        self.args.v_head_dim
        )

    if self.layer_id < self.args.n_dense_layers:
      self.ffn = MLP(self.args.dim, self.args.inter_dim)

    self.ffn = MixtureOfExperts(
        self.args.dim,
        self.args.routed_experts,
        self.args.n_activated_experts,
        self.inter_dim
        )
    self.attn_norm = nn.RMSNorm(self.args.dim)
    self.ffn_norm = nn.RMSNorm(self.args.dim)


  @nn.compact
  def __call__(self, x: jnp.ndarray, start: int, freq_cis: jnp.ndarray, mask: Optional[jnp.ndarray] = None):
    """
    Forward pass for the Transformer Block
    """

    x = x + self.attention(self.attn_norm(x), start, freq_cis, mask)
    x = x + self.ffn(self.ffn_norm(x))
    return x



# KDA Layer

class KDA_Layer(nn.Module):
  """
  Kimi Delta Attention Layer
  """
  args: ModelArg

  def setup(self):
    self.conv: ShortConv = ShortConv()
    self.kda = KDA(self.args.dim)
    self.norm = RMSNorm(dim=self.args.dim)
    self.kernel_size: int = 4
    self.first: bool = True

    pass

  def __call__(self, x: jnp.ndarray, initial_state: Optional[jnp.ndarray] = None) -> Any:
    b, t, h = x.shape

    x_norm = self.norm(x)

    batch, seq, dim = x.shape

    weights = self.param(
        'weight',
        nn.initializers.zeros,
         (dim, self.kernel_size)
         )

    bias = self.param(
        'bias',
        nn.initializers.zeros,
         (dim,)
         )

    conv = self.conv(x=x, weights=weights, bias=bias)

    q, k = jax.numpy.linalg.norm(nn.swish(conv)), jax.numpy.linalg.norm(nn.swish(conv))

    v = nn.swish(conv)

    b = nn.sigmoid(nn.Dense(h)(x))

    def low_rank_proj(x, activation_fn):
          """shared structure for α and output gate"""
          x = nn.Dense(self.head_dim, use_bias=False)(x)
          x = nn.Dense(self.heads * self.head_dim, use_bias=False)(x)
          return activation_fn(x)

    alpha = low_rank_proj(x, nn.log_sigmoid)
    gate  = low_rank_proj(x, nn.sigmoid)


    if self.first:
      self.first = False
      KDA_out, S = self.kda(q, k, v, alpha, b)
    else:
      KDA_out, S = self.kda(q, k, v, alpha, b, initial_state)

    dot_ = KDA_out * gate

    linear_out = Linear(self.args.in_feat,
                    self.args.out_feat,
                    weights=weights,
                    bias=bias)

    out = linear_out(dot_)

    return out + x_norm


# MoE Layer
class MoE_Layer(nn.Module):
  """
  Mixture of Experts Layer = MoE + Norm
  """

  args: ModelArg

  def setup(self):
    self.norm = RMSNorm(self.args.dim)
    self.moe = MixtureOfExperts(dim=self.args.dim,
                                n_routed_experts=self.args.n_routed_experts,
                                n_groups=self.args.n_groups,
                                n_activated_experts=self.args.n_activated_experts,
                                n_shared_experts=self.args.shared_experts,
                                topk_groups=self.args.topk_groups,
                                inter_dim=self.args.inter_dim,
                                route_scale=self.args.route_scale,
                                score_func=self.args.score_func
                                )

  @nn.compact
  def __call__(self, x: jnp.ndarray):
    """
    Forward pass for the Mixture of Experts Layer
    """

    assert x.ndim == 3

    x_norm = self.norm(x)

    out = self.moe(x_norm)

    return x + out



# MLA Layer

class MLA_Layer(nn.Module):
  """
  Multi-Layer Attention Layer
  """
  args: ModelArg



  def setup(self):
    self.norm = RMSNorm(self.args.dim)
    self.mla = multihead_attn(dim=self.args.dim,
                              heads=self.args.heads,
                              local_heads=self.args.local_heads,
                              q_lora_rank=self.args.q_lora_rank,
                              kv_lora_rank=self.args.kv_lora_rank,
                              v_head_dim=self.args.v_head_dim)


  @nn.compact
  def __call__(self, x: jnp.ndarray):
    """
    Forward pass for the Multi-Layer Attention Layer
    """

    x_norm = self.norm(x)

    out = self.mla(x_norm)

    return x + out


# One (MLA Layer + MoE Layer) for every 3 (KDA Layer + MoE Layer)

class Transformer(nn.Module):

  """
  Transformer Model with positional embedding, multiple layers and out
  """
  args: ModelArg

  def setup(self):
    self.max_seq = self.args.max_seq
    self.layers: list[Block] = []
    for layer in range(self.args.n_layers):
      self.layers.append(Block(layer_id=layer, args=self.args))
    self.norm = RMSNorm(self.args.dim)
    self.head = Linear(self.args.dim, self.args.vocab_size, dtype=jnp.float32)


    pass


  @nn.compact
  def __call__(self, tokens: jnp.ndarray, start: int = 0):
    """
    Forward pass for the Transformer model
    """

    assert tokens.shape


    pass








SyntaxError: invalid decimal literal (2742856162.py, line 160)

In [ ]:
ss

# ⚡ Kernels



In [ ]:


# BF16 Kernel



#

# Attention Residual Plugin







# Kernels


Quantisization kernels, for faster training

In [ ]:
from jax.experimental import pallas as pl


def make_act_q_kernel(block_size: int, scale_fmt = None):


  def act_quant_kernel(x_ref, y_ref, s_ref):
    """
    Quantizes the input tensor `x_ref` and stores the result in `y_ref` and the scaling factor in `s_ref`.

    """

    pid = pl.program_id(axis=0)

    x = x_ref[pid, :].astype(jnp.float32)

    amax = jnp.max(jnp.abs(x))

    amax = jnp.maximum(amax, 1e-4)

    s = amax / 448.0

    if scale_fmt == 'ue8m0':
      exp = jnp.ceil(jnp.log2(s))
      s = jnp.exp2(exp)

    y = x / s

    # store
    y_ref[pid, :] = y.astype(jnp.float8_e4m3fn)

    s_ref[pid] = s




  return act_quant_kernel



def act_quant(x: jnp.ndarray, block_size: int, scale_fmt: Optional[str] = None) -> Tuple[jnp.ndarray, jnp.ndarray]:


  assert x.shape[-1] % block_size == 0, f'Last dimension size must be divisible by block_size (block_size={block_size})'
  y = pl.empty_like(x, dtype=pl.float8_e4m3fn)
  # TODO: allocate scale output array (x.new_empty is PyTorch-only)
  s = jnp.empty((*x.shape[:-1], x.shape[-1] // block_size), dtype=jnp.float32)




# Training


# Benchmark